[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_72_Structured_Logging_Trace_Search.ipynb)

# Lesson 72 — Structured Logging & Trace Search
### Phase 8: Production Ops for LLM Systems · Lesson 2 of ~6

Welcome back, Gourav. Last lesson (**L71**) you built the three pillars of observability into an `observability/` module: **traces** (spans → JSONL with OpenTelemetry GenAI attribute names), **metrics** (p50/p95/p99 rollups), and **online evals** (heuristic + LLM-judge scoring on a sample, plus a drift monitor). Those tools answer *"is something wrong, and how wrong?"* — in **aggregate**.

Today answers a different question. It's 2am. You get paged: **"agent runs for user_042 are failing."** Aggregate metrics told you the error rate ticked up. They will **not** tell you *which* requests, for *whom*, in *what order*, or *why*. For that you need the **query layer** of observability: **structured logging** and **trace search**.

> **The one-line distinction from L71:** aggregate metrics tell you *something* is wrong; trace search tells you *what* is wrong, for *whom*, and lets you *read the story* of a single failing request end-to-end.

#### Phase 8 roadmap (tentative — adapts to your questions)

| Lesson | Topic | Status |
|---|---|---|
| L71 | Observability & online evals (3 pillars) | ✅ done |
| **L72** | **Structured logging & trace search** | **← today** |
| L73 | Alerting, SLOs & on-call | next |
| L74 | A/B testing & guarded rollouts | planned |
| L75 | Feedback loops & the data flywheel | planned |
| L76 | Capstone: ship a prod-obs module for `agent-bench` | planned |

#### What you'll build today
A small, portable **investigation toolkit** you can drop onto any agent:
1. A **structured logger** — every log line is a queryable JSON *event*, not a formatted string.
2. **Correlation IDs** + **context propagation** with `contextvars` — the request/trace/session/user IDs that stitch spans and logs together, attached **automatically** so you never thread them through function arguments.
3. A **`TraceStore`** search index — query by user, session, status, time window, latency, cost, and free text.
4. The **payoff query**: *"show me every failed request from user_042 in the last hour."*
5. **Session reconstruction** — replay a full multi-turn agent session in order.
6. **PII redaction** — scrub secrets before they ever hit disk (folds in L71's homework).

Everything runs on **deterministic simulated traffic** — no API key, no network.

## 1. The pager scenario: why aggregates aren't enough

L71's metrics table said: *error rate 6%, p99 latency 4.1s, quality dropped 0.2 below baseline.* Useful — it's how you **know** to look. But it's a summary. A summary can't be clicked into.

To debug, you need to move from **"how many"** to **"which ones"**:

| | L71 — aggregate observability | L72 — investigative observability |
|---|---|---|
| **Question** | *Is something wrong?* | *What exactly is wrong, for whom?* |
| **Unit** | the fleet (all traffic rolled up) | one request / one session |
| **Output** | a number (error rate, p99, drift alarm) | a **list of matching records** you can read |
| **Data shape** | counters & histograms | **searchable structured events** |
| **Trigger** | dashboards, thresholds | a page, a support ticket, a user complaint |
| **Key tool** | `Metrics.rollup()` | `TraceStore.search(...)` |
| **Failure if missing** | you're flying blind | you *know* it's broken but can't find it |

They are two halves of one loop: **metrics detect, search diagnoses.**

#### The three IDs that make search possible

A single agent run is not one event — it's a fetch, an LLM call, a tool call, a second LLM call... To reassemble them you attach shared identifiers:

- **`trace_id`** — one **agent run**, end to end. Every span in that run shares it. *("This whole distillation.")*
- **`span_id`** — one **step** inside the run (the fetch, the LLM call). Has a `parent_id` pointing up the call tree. *("Just the LLM call.")*
- **`session_id`** — one **conversation / user session** spanning *many* runs. *("Everything user_042 did this afternoon.")*
- plus **`user_id`** and **`request_id`** for filtering.

Without these, your logs are a shuffled pile of sentences. With them, they're a **queryable graph**. The rest of the lesson is: attach them automatically, then search on them.

In [ ]:
# === Setup (Colab-friendly, no API key required) =====================
# Everything today is deterministic simulated traffic — nothing calls a live API.
# We install `rich` only for pretty search-result tables.
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import rich  # noqa
except ImportError:
    _pip("rich")

import os, json, uuid, re, time, random, contextvars, io
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field, asdict
from contextlib import contextmanager
from rich.console import Console
from rich.table import Table

# Force plain output so this renders identically in Colab and in headless CI.
console = Console(force_jupyter=False, no_color=True, highlight=False, width=100)

# Where trace/log files land. Defaults to Colab's /content; falls back locally.
RUN_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
LOG_PATH = os.path.join(RUN_DIR, "events.jsonl")
print("Python:", sys.version.split()[0])
print("Writing events to:", LOG_PATH)

# A FIXED 'now' so time-window queries ('last hour') are 100% reproducible.
# In real code this would be datetime.now(timezone.utc).
NOW = datetime(2026, 7, 18, 12, 0, 0, tzinfo=timezone.utc)
print("Simulated wall-clock NOW =", NOW.isoformat())

## 2. Structured logging: events, not sentences

Here is the single most important habit shift for production LLM systems.

**String logging** (what most of us start with):
```python
print(f"error handling request for {user} on {url}: {err}")
```
That line is *readable by a human* and *useless to a machine*. To find "all errors for user_042" you'd have to `grep` and parse free text with regex forever. The user ID, the URL, the error — all melted into one string.

**Structured logging** — the log record is a **dict** with named fields:
```python
log.error("request_failed", user_id=user, url=url, error=str(err))
```
Now every field is independently queryable. "All errors for user_042" is a filter on two fields, not a regex safari. The human-readable message (`"request_failed"`) becomes just *one field among many* — a stable **event name** you can group by.

The rules of good structured logging:
- **Message = a short, stable event name** (`"llm_call_completed"`), not an interpolated sentence. Stable names are groupable and alertable.
- **Everything variable goes in fields** (`user_id`, `latency_ms`, `model`), never in the message string.
- **One line = one JSON object** (JSONL). Append-only, trivially streamable, every tool can read it.
- **Always include the correlation IDs** (next section) so a line can be traced back to its request.

In [ ]:
# === A minimal structured logger ====================================
LEVELS = {"DEBUG": 10, "INFO": 20, "WARNING": 30, "ERROR": 40}

class StructuredLogger:
    '''Emits one JSON object per log record to a JSONL sink.

    A record is: timestamp + level + event-name + arbitrary fields.
    The message is a STABLE EVENT NAME, never an interpolated sentence.
    (Correlation IDs get auto-attached in the next section via `_context_provider`.)
    '''
    def __init__(self, sink, min_level="DEBUG", clock=None, context_provider=None):
        self.sink = sink                       # a writable file-like object
        self.min_level = LEVELS[min_level]
        self.clock = clock or (lambda: datetime.now(timezone.utc))
        # A hook that returns dict of correlation IDs; wired up in section 3.
        self._context_provider = context_provider or (lambda: {})

    def _emit(self, level, event, **fields):
        if LEVELS[level] < self.min_level:
            return
        record = {
            "ts": self.clock().isoformat(),
            "level": level,
            "event": event,               # stable, groupable name
            **self._context_provider(),   # trace_id / session_id / user_id / ...
            **fields,                     # everything variable
        }
        self.sink.write(json.dumps(record) + "\n")
        self.sink.flush()
        return record

    def debug(self, event, **f):   return self._emit("DEBUG", event, **f)
    def info(self, event, **f):    return self._emit("INFO", event, **f)
    def warning(self, event, **f): return self._emit("WARNING", event, **f)
    def error(self, event, **f):   return self._emit("ERROR", event, **f)

# --- self-test: a record is structured, timestamped, and level-filtered ---
_buf = io.StringIO()
_log = StructuredLogger(_buf, min_level="INFO", clock=lambda: NOW)
_log.debug("noisy_trace", detail="should be filtered out")   # below INFO -> dropped
rec = _log.info("request_failed", user_id="user_042", url="/paper/123", error="timeout")
line = _buf.getvalue().strip()
parsed = json.loads(line)
assert parsed["event"] == "request_failed"
assert parsed["user_id"] == "user_042"      # a FIELD, not buried in a string
assert parsed["level"] == "INFO"
assert "noisy_trace" not in _buf.getvalue() # DEBUG was filtered below min_level
print("One JSONL record:\n", line)
print("\nField access (the whole point):", parsed["user_id"], "|", parsed["error"])
# 💡 EXPERIMENT: change min_level to "ERROR" — the info() line disappears too.

## 3. Correlation IDs & context propagation with `contextvars`

Here's the trap. You *know* every log line needs `trace_id`, `session_id`, `user_id`. The naive fix is to pass them as arguments:

```python
def fetch(url, trace_id, session_id, user_id): ...
def llm_call(prompt, trace_id, session_id, user_id): ...   # threaded through EVERYTHING
```

This is miserable and fragile — every function signature bloats, and the day someone forgets to pass one, a log line becomes an orphan you can't correlate.

The clean solution is **context propagation**. You set the request's identity **once** at the top of the request, into an **ambient context**, and every span and log line reads it from there automatically. Python's tool for this is **`contextvars`** — think of it as "a variable scoped to the current logical thread of execution."

**Why `contextvars` and not a global, or `threading.local`?**
- A plain global would be shared across *all* concurrent requests — user_042's ID would leak into user_099's logs. (Remember L67's cross-task contamination bug? Same class of danger.)
- `threading.local` is per-OS-thread, but async code (L57/L67) runs many requests on *one* thread. `contextvars` is the one primitive that is correct under **both** threads **and** `asyncio` — each task gets its own isolated copy. It's exactly what OpenTelemetry uses under the hood.

In [ ]:
# === Ambient request context via contextvars =========================
_current_ctx = contextvars.ContextVar("request_ctx", default=None)

@dataclass
class RequestContext:
    trace_id: str
    session_id: str
    user_id: str
    request_id: str

def current_context():
    '''Return the active correlation IDs as a dict (empty if none set).

    This is the hook we pass to StructuredLogger — so EVERY log line
    auto-attaches the current request's IDs without us threading them.'''
    ctx = _current_ctx.get()
    return asdict(ctx) if ctx else {}

@contextmanager
def request_scope(user_id, session_id, request_id=None, trace_id=None):
    '''Open a scope. Everything logged inside inherits these IDs automatically.

    contextvars.set() returns a token; resetting it on exit restores whatever
    context was active before — so nested/concurrent scopes never leak.'''
    ctx = RequestContext(
        trace_id=trace_id or "trace_" + uuid.uuid4().hex[:12],
        session_id=session_id,
        user_id=user_id,
        request_id=request_id or "req_" + uuid.uuid4().hex[:8],
    )
    token = _current_ctx.set(ctx)
    try:
        yield ctx
    finally:
        _current_ctx.reset(token)   # <-- the isolation guarantee

# --- self-test: IDs flow into logs WITHOUT being passed explicitly ---
_buf2 = io.StringIO()
log = StructuredLogger(_buf2, clock=lambda: NOW, context_provider=current_context)

log.info("outside_any_scope")                       # no IDs available yet
with request_scope(user_id="user_042", session_id="sess_A") as ctx:
    log.info("work_started", step="fetch")          # note: we pass NO ids here
    with request_scope(user_id="user_099", session_id="sess_B"):
        log.warning("nested_scope", note="different user")
    log.info("back_outside_nested", step="llm_call")

records = [json.loads(l) for l in _buf2.getvalue().splitlines()]
assert "user_id" not in records[0]                  # outside scope -> orphan (no ids)
assert records[1]["user_id"] == "user_042"          # auto-attached!
assert records[2]["user_id"] == "user_099"          # nested scope isolated
assert records[3]["user_id"] == "user_042"          # restored after nested exit
print("Context propagation working. Each line's user_id (None = outside scope):")
for r in records:
    print(f"  {r['event']:<22} user_id={r.get('user_id')}  session={r.get('session_id')}")
# 💡 EXPERIMENT: log.info() inside a scope but comment out context_provider in
# the constructor above — watch the IDs vanish. That's what threading them by
# hand protects against, and what contextvars gives you for free.

## 4. Instrument a mini agent & generate findable traffic

Now we combine everything: a small **`Tracer`** (carried forward and condensed from L71 — spans with OTel GenAI attribute names) that *also* reads the ambient context and *also* emits a structured log line per span. One instrumentation, three outputs: a trace, correlated logs, and searchable fields.

Then we simulate a `paper-distiller`-shaped workload. Crucially, we **bake in a findable incident** so the search in section 5 has a real needle:
- ~200 requests across several users and sessions, timestamped over the **last ~3 hours**.
- **`user_042` gets a burst of failures inside the last hour** — the exact thing the pager complained about.
- Some records carry **PII** (an email, an API key) so section 6's redaction has something to scrub.

Everything is seeded and deterministic.

In [ ]:
# === Tracer that auto-correlates + logs, then generate traffic =======
@dataclass
class Span:
    name: str
    span_id: str
    trace_id: str
    parent_id: str = None
    start: float = 0.0
    end: float = 0.0
    status: str = "OK"
    attributes: dict = field(default_factory=dict)
    @property
    def latency_ms(self): return round((self.end - self.start) * 1000, 2)

class Tracer:
    '''Minimal tracer: one span per step, JSONL sink, OTel GenAI attr names,
    auto-attaches the ambient RequestContext, and emits a log line per span.'''
    def __init__(self, trace_sink, logger, clock):
        self.trace_sink = trace_sink
        self.logger = logger
        self.clock = clock          # returns float seconds (monotonic-ish)
        self._stack = []

    @contextmanager
    def span(self, name, **attributes):
        ctx = _current_ctx.get()
        trace_id = ctx.trace_id if ctx else "trace_orphan"
        parent = self._stack[-1].span_id if self._stack else None
        sp = Span(name=name, span_id="span_" + uuid.uuid4().hex[:10],
                  trace_id=trace_id, parent_id=parent,
                  start=self.clock(), attributes=dict(attributes))
        self._stack.append(sp)
        try:
            yield sp
        except Exception as e:
            sp.status = "ERROR"; sp.attributes["error"] = str(e); raise
        finally:
            sp.end = self.clock()
            self._stack.pop()
            # 1) write the raw span to the trace file
            self.trace_sink.write(json.dumps({
                "kind": "span", "name": sp.name, "span_id": sp.span_id,
                "trace_id": sp.trace_id, "parent_id": sp.parent_id,
                "status": sp.status, "latency_ms": sp.latency_ms,
                **sp.attributes,
            }) + "\n"); self.trace_sink.flush()
            # 2) emit a correlated structured log line (ctx auto-attached)
            self.logger.info("span_completed", span_name=sp.name,
                             span_id=sp.span_id, status=sp.status,
                             latency_ms=sp.latency_ms,
                             **{k: v for k, v in sp.attributes.items()
                                if k in ("model", "error", "cost_usd")})

# ---- generate deterministic, incident-laden traffic -----------------
random.seed(72)
USERS = [f"user_{i:03d}" for i in (11, 27, 42, 58, 99)]
DRIFT_USER = "user_042"                       # the pager's villain
sink = open(LOG_PATH, "w")                    # ALL events (spans + logs) share this file
logger = StructuredLogger(sink, context_provider=current_context,
                          clock=lambda: NOW)   # placeholder; overridden per-request below

def simulate(n=200):
    for i in range(n):
        # spread requests over the last ~3 hours; user_042's failures cluster
        # in the LAST hour so a 'last hour' query isolates the incident.
        minutes_ago = random.randint(0, 179)
        ts = NOW - timedelta(minutes=minutes_ago)
        user = DRIFT_USER if (minutes_ago < 55 and i % 3 == 0) else random.choice(USERS)
        session = f"sess_{user}_{minutes_ago // 60}"   # ~1 session per user per hour
        # user_042 in the last hour fails hard; everyone else mostly succeeds.
        in_incident = (user == DRIFT_USER and minutes_ago < 55)
        fails = in_incident or (random.random() < 0.03)
        # a per-request clock so each span gets this request's timestamp + a latency
        base = ts.timestamp()
        step = {"t": base}
        def clk():
            step["t"] += random.uniform(0.05, 0.9) + (2.5 if fails else 0.0)
            return step["t"]
        logger.clock = lambda ts=ts: ts       # log timestamps = request time
        tracer = Tracer(sink, logger, clk)
        # some requests carry PII we must scrub later
        pii = {"contact_email": "gourav@example.com", "api_key": "sk-live-ABC123XYZ"} \
              if i % 25 == 0 else {}
        with request_scope(user_id=user, session_id=session):
            try:
                with tracer.span("distill_paper", paper_id=f"p{i:04d}", **pii):
                    with tracer.span("fetch", url=f"/paper/p{i:04d}"):
                        if fails and random.random() < 0.5:
                            raise TimeoutError("upstream fetch timed out")
                    with tracer.span("llm_call",
                                     **{"gen_ai.system": "anthropic",
                                        "gen_ai.request.model": "claude-sonnet-5",
                                        "model": "claude-sonnet-5",
                                        "gen_ai.usage.input_tokens": random.randint(400, 900),
                                        "gen_ai.usage.output_tokens": random.randint(80, 300),
                                        "cost_usd": round(random.uniform(0.002, 0.02), 5)}):
                        if fails:
                            raise RuntimeError("model returned malformed JSON")
            except Exception:
                pass   # failure is recorded in the span/log; we keep serving traffic

simulate(200)
sink.close()
n_lines = sum(1 for _ in open(LOG_PATH))
print(f"Wrote {n_lines} events (spans + logs) to {LOG_PATH}")
print("Sample line:", open(LOG_PATH).readline().strip()[:120], "...")

## 5. Trace search: the query layer

We now have a JSONL file mixing raw spans and structured logs — a haystack. `grep` can find a substring, but it can't answer *"failed requests for user_042 **in the last hour** sorted by latency."* For that we load the events into a tiny **`TraceStore`** that indexes them and exposes **composable filters**.

Real systems use Elasticsearch / Loki / Datadog / Langfuse for this. The API you'll build is a faithful miniature of theirs: **load once, then chain filters**. The point isn't the storage engine — it's learning to *think in queries over structured events* instead of grepping strings.

In [ ]:
# === A searchable store over the event log ===========================
class Query:
    '''A chainable view over a list of event dicts. Each filter returns a
    new Query, so you compose: store.logs().user('u').failed().last(60).'''
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __iter__(self): return iter(self.rows)

    def user(self, uid):     return Query([r for r in self.rows if r.get("user_id") == uid])
    def session(self, sid):  return Query([r for r in self.rows if r.get("session_id") == sid])
    def trace(self, tid):    return Query([r for r in self.rows if r.get("trace_id") == tid])
    def event(self, name):   return Query([r for r in self.rows if r.get("event") == name])
    def failed(self):        return Query([r for r in self.rows if r.get("status") == "ERROR"
                                           or r.get("level") == "ERROR"])
    def slower_than(self, ms):
        return Query([r for r in self.rows if (r.get("latency_ms") or 0) > ms])
    def search_text(self, needle):
        n = needle.lower()
        return Query([r for r in self.rows if n in json.dumps(r).lower()])
    def since(self, dt):
        return Query([r for r in self.rows
                      if r.get("ts") and datetime.fromisoformat(r["ts"]) >= dt])
    def last(self, minutes, now=None):
        return self.since((now or NOW) - timedelta(minutes=minutes))
    def sort_by(self, key, reverse=True):
        return Query(sorted(self.rows, key=lambda r: r.get(key) or 0, reverse=reverse))

class TraceStore:
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path) if l.strip()]
    def all(self):   return Query(self.rows)
    def logs(self):  return Query([r for r in self.rows if "level" in r])   # log records
    def spans(self): return Query([r for r in self.rows if r.get("kind") == "span"])

store = TraceStore(LOG_PATH)
print(f"Loaded {len(store.all())} events -> "
      f"{len(store.logs())} log records, {len(store.spans())} spans")

# --- self-test: filters compose and narrow correctly ---
all_042 = store.logs().user("user_042")
failed_042 = all_042.failed()
assert len(failed_042) <= len(all_042)
assert all(r["user_id"] == "user_042" for r in failed_042)
# .failed() matches status=="ERROR" (a failed span logged at INFO) OR level=="ERROR".
# Failed spans arrive as span_completed@INFO with status="ERROR" — so we check either.
assert all(r.get("status") == "ERROR" or r.get("level") == "ERROR" for r in failed_042)
print(f"user_042 has {len(all_042)} log records, {len(failed_042)} of them failures.")
# 💡 EXPERIMENT: chain a third filter — .user('user_042').failed().slower_than(2000)

## 6. The payoff: *"every failed request from user_042 in the last hour"*

This is the exact sentence from the 2am page. With structured events + correlation IDs + a query layer, it's now **one readable line of code** — not an afternoon of grepping. Watch the filters compose left-to-right, each narrowing the set.

In [ ]:
# === The investigation query =========================================
hits = (store.logs()
             .user("user_042")     # who the pager named
             .failed()             # only errors
             .last(60)             # in the last hour (vs NOW)
             .sort_by("latency_ms"))   # worst first

print(f"Found {len(hits)} failed requests for user_042 in the last hour.\n")

tbl = Table(title="user_042 · failures · last 60 min", show_lines=False)
for c in ("time", "event", "span", "latency_ms", "trace_id"):
    tbl.add_column(c)
for r in list(hits)[:10]:
    tbl.add_row(r["ts"][11:19], r["event"], r.get("span_name", "-"),
                str(r.get("latency_ms", "-")), (r.get("trace_id") or "-")[:16])
console.print(tbl)

# assertions: the injected incident is found AND the query is specific
assert len(hits) > 0, "should surface the injected user_042 incident"
older_fails = store.logs().user("user_042").failed().rows
recent = [r for r in older_fails
          if datetime.fromisoformat(r["ts"]) >= NOW - timedelta(minutes=60)]
assert len(hits) == len(recent)                      # 'last hour' filter is exact
# specificity: no OTHER user leaks into this result set
assert all(r["user_id"] == "user_042" for r in hits)
# and pick ONE failing trace to read end-to-end (the drill-down move)
worst = list(hits)[0]
print(f"\nDrill into worst trace {worst['trace_id']}:")
for r in store.all().trace(worst["trace_id"]).rows:
    tag = r.get("event") or r.get("name")
    print(f"  [{r.get('level','SPAN'):<5}] {tag:<16} status={r.get('status','-')}"
          f" latency={r.get('latency_ms','-')}")

## 7. Session reconstruction: replay the whole conversation

A single failing trace tells you about *one run*. But agents are conversational — the *real* story is often the **session**: everything a user did across many runs. `session_id` is the join key. Grouping every event by session and ordering by time gives you a **timeline you can replay** — which is how you answer *"what was user_042 doing right before it started failing?"*

In [ ]:
# === Reconstruct a session as an ordered timeline ====================
def reconstruct_session(store, session_id):
    rows = store.all().session(session_id).rows
    # order by timestamp; log records have ts, spans inherit via their log twin,
    # so we key spans by their matching log line's time when present.
    return sorted(rows, key=lambda r: r.get("ts") or "")

# find user_042's busiest session in the incident window
from collections import Counter
sess_counts = Counter(r.get("session_id") for r in store.logs().user("user_042").rows
                      if r.get("session_id"))
target_session = sess_counts.most_common(1)[0][0]
timeline = reconstruct_session(store, target_session)

print(f"Session {target_session}: {len(timeline)} events\n")
tl = Table(title=f"Session replay · {target_session}")
for c in ("time", "level", "event", "status"):
    tl.add_column(c)
for r in timeline[:12]:
    tl.add_row(r.get("ts", "-")[11:19], r.get("level", "SPAN"),
               r.get("event") or r.get("name", "-"), str(r.get("status", "-")))
console.print(tl)

# assertions: the timeline is complete and correctly ordered
assert all(r.get("session_id") == target_session for r in timeline if "session_id" in r)
times = [r["ts"] for r in timeline if r.get("ts")]
assert times == sorted(times), "session timeline must be chronologically ordered"
distinct_traces = {r.get("trace_id") for r in timeline if r.get("trace_id")}
print(f"\nSession spans {len(distinct_traces)} distinct agent run(s) / trace(s).")
# 💡 EXPERIMENT: reconstruct a HEALTHY user's session (e.g. user_011's) and
# compare the shape — same structure, no ERROR lines.

## 8. PII redaction: never log a secret

Structured logs are wonderful *and* dangerous: it is trivially easy to dump an API key, a bearer token, or a user's email straight into a searchable store where it lives forever and is visible to everyone with dashboard access. This was **L71's homework #1**; here we make it real.

The fix is a **redaction filter on the write path** — it runs *before* the record touches disk, so a secret is scrubbed even if a developer accidentally logs it. Two complementary strategies:
- **Field denylist** — known-sensitive keys (`api_key`, `password`, `authorization`) are replaced wholesale.
- **Value patterns** — regexes that catch secrets *anywhere* in the values (`sk-...` keys, `Bearer ...` tokens, emails), so PII pasted into a free-text field is still caught.

Redact at the boundary, not at every call site — one filter, no reliance on every developer remembering.

In [ ]:
# === Redaction filter, wired into the logger's write path ============
DENY_KEYS = {"api_key", "password", "authorization", "secret", "token"}
PATTERNS = [
    (re.compile(r"sk-[A-Za-z0-9\-]{6,}"), "[REDACTED_KEY]"),
    (re.compile(r"Bearer\s+[A-Za-z0-9\.\-_]+"), "Bearer [REDACTED]"),
    (re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}"), "[REDACTED_EMAIL]"),
]

def redact(record):
    '''Return a copy of the record with sensitive fields/values scrubbed.'''
    clean = {}
    for k, v in record.items():
        if k in DENY_KEYS:
            clean[k] = "[REDACTED]"
            continue
        if isinstance(v, str):
            for pat, repl in PATTERNS:
                v = pat.sub(repl, v)
        clean[k] = v
    return clean

class RedactingLogger(StructuredLogger):
    '''Same as StructuredLogger, but scrubs every record before writing.'''
    def _emit(self, level, event, **fields):
        if LEVELS[level] < self.min_level:
            return
        record = {"ts": self.clock().isoformat(), "level": level, "event": event,
                  **self._context_provider(), **fields}
        record = redact(record)                 # <-- the one added line
        self.sink.write(json.dumps(record) + "\n"); self.sink.flush()
        return record

# --- prove PII goes in and NEVER lands on disk ---
_buf3 = io.StringIO()
rlog = RedactingLogger(_buf3, clock=lambda: NOW)
rlog.info("user_login", api_key="sk-live-SUPERSECRET999",
          note="reach me at gourav@example.com or Bearer abc.def.ghi")
out = _buf3.getvalue()
assert "SUPERSECRET" not in out and "sk-live" not in out   # denylisted key scrubbed
assert "gourav@example.com" not in out                     # email pattern scrubbed
assert "Bearer abc" not in out                             # token pattern scrubbed
assert "[REDACTED" in out                                  # replacement markers present
print("Redacted record on disk:\n", out.strip())
# 💡 EXPERIMENT: add a credit-card regex \b(?:\d[ -]*?){13,16}\b to PATTERNS.

In [ ]:
# === Write it all to a reusable observability/ module ================
import os, importlib.util
os.makedirs(os.path.join(RUN_DIR, 'observability'), exist_ok=True)
module_path = os.path.join(RUN_DIR, 'observability', 'logging_search.py')

# The module source is a RAW triple-quoted string so regex backslashes and
# the newline write-literal survive verbatim into the written .py file.
MODULE_SRC = r"""
'''observability.logging_search - structured logging, context propagation,
redaction, and a searchable TraceStore. Built in Lesson 72.'''
import json, uuid, re, contextvars
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from contextlib import contextmanager

LEVELS = {"DEBUG": 10, "INFO": 20, "WARNING": 30, "ERROR": 40}
_current_ctx = contextvars.ContextVar("request_ctx", default=None)

@dataclass
class RequestContext:
    trace_id: str; session_id: str; user_id: str; request_id: str

def current_context():
    c = _current_ctx.get()
    return asdict(c) if c else {}

@contextmanager
def request_scope(user_id, session_id, request_id=None, trace_id=None):
    ctx = RequestContext(
        trace_id=trace_id or "trace_" + uuid.uuid4().hex[:12],
        session_id=session_id, user_id=user_id,
        request_id=request_id or "req_" + uuid.uuid4().hex[:8])
    token = _current_ctx.set(ctx)
    try:
        yield ctx
    finally:
        _current_ctx.reset(token)

DENY_KEYS = {"api_key", "password", "authorization", "secret", "token"}
PATTERNS = [
    (re.compile(r"sk-[A-Za-z0-9\-]{6,}"), "[REDACTED_KEY]"),
    (re.compile(r"Bearer\s+[A-Za-z0-9\.\-_]+"), "Bearer [REDACTED]"),
    (re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}"), "[REDACTED_EMAIL]"),
]

def redact(record):
    clean = {}
    for k, v in record.items():
        if k in DENY_KEYS:
            clean[k] = "[REDACTED]"; continue
        if isinstance(v, str):
            for pat, repl in PATTERNS:
                v = pat.sub(repl, v)
        clean[k] = v
    return clean

class StructuredLogger:
    def __init__(self, sink, min_level="DEBUG", clock=None,
                 context_provider=None, do_redact=True):
        self.sink = sink; self.min_level = LEVELS[min_level]
        self.clock = clock or (lambda: datetime.now(timezone.utc))
        self._context_provider = context_provider or (lambda: {})
        self.do_redact = do_redact
    def _emit(self, level, event, **fields):
        if LEVELS[level] < self.min_level:
            return
        record = {"ts": self.clock().isoformat(), "level": level, "event": event,
                  **self._context_provider(), **fields}
        if self.do_redact:
            record = redact(record)
        self.sink.write(json.dumps(record) + "\n"); self.sink.flush()
        return record
    def debug(self, e, **f):   return self._emit("DEBUG", e, **f)
    def info(self, e, **f):    return self._emit("INFO", e, **f)
    def warning(self, e, **f): return self._emit("WARNING", e, **f)
    def error(self, e, **f):   return self._emit("ERROR", e, **f)

class Query:
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __iter__(self): return iter(self.rows)
    def user(self, u):    return Query([r for r in self.rows if r.get("user_id") == u])
    def session(self, s): return Query([r for r in self.rows if r.get("session_id") == s])
    def trace(self, t):   return Query([r for r in self.rows if r.get("trace_id") == t])
    def event(self, n):   return Query([r for r in self.rows if r.get("event") == n])
    def failed(self):     return Query([r for r in self.rows
                          if r.get("status") == "ERROR" or r.get("level") == "ERROR"])
    def slower_than(self, ms):
        return Query([r for r in self.rows if (r.get("latency_ms") or 0) > ms])
    def search_text(self, needle):
        n = needle.lower()
        return Query([r for r in self.rows if n in json.dumps(r).lower()])
    def since(self, dt):
        return Query([r for r in self.rows
                      if r.get("ts") and datetime.fromisoformat(r["ts"]) >= dt])
    def last(self, minutes, now):
        return self.since(now - timedelta(minutes=minutes))
    def sort_by(self, key, reverse=True):
        return Query(sorted(self.rows, key=lambda r: r.get(key) or 0, reverse=reverse))

class TraceStore:
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path) if l.strip()]
    def all(self):   return Query(self.rows)
    def logs(self):  return Query([r for r in self.rows if "level" in r])
    def spans(self): return Query([r for r in self.rows if r.get("kind") == "span"])
"""

with open(module_path, 'w') as f:
    f.write(MODULE_SRC)

# import it back and smoke-test the real module
spec = importlib.util.spec_from_file_location('logging_search', module_path)
ls = importlib.util.module_from_spec(spec); spec.loader.exec_module(ls)

_b = io.StringIO()
mlog = ls.StructuredLogger(_b, context_provider=ls.current_context, clock=lambda: NOW)
with ls.request_scope(user_id='user_042', session_id='sess_Z'):
    mlog.error('boom', api_key='sk-live-XYZ', detail='mail me at a@b.com')
line = json.loads(_b.getvalue())
assert line['user_id'] == 'user_042'          # context propagation works
assert line['api_key'] == '[REDACTED]'        # redaction works
assert 'a@b.com' not in _b.getvalue()         # email pattern works
mstore = ls.TraceStore(LOG_PATH)              # store reads our earlier traffic
assert len(mstore.logs().user('user_042').failed()) > 0
print('Module written to', module_path)
print('Smoke test passed: context + redaction + TraceStore all import cleanly.')

## 9. Pitfalls (the ones that bite in production)

| # | Pitfall | Why it hurts | Fix |
|---|---|---|---|
| 1 | **String logs, not structured** | `grep`-and-regex forever; can't filter by field | message = stable event name; everything else in fields |
| 2 | **Threading IDs through every function** | signature bloat; one miss = an orphan line | `contextvars` + a `request_scope` context manager |
| 3 | **Using a global or `threading.local` for context** | leaks across concurrent/async requests (L67-style contamination) | `contextvars` — correct under threads *and* `asyncio` |
| 4 | **Forgetting to reset the context token** | the previous request's IDs bleed into the next | always `_current_ctx.reset(token)` in a `finally` |
| 5 | **Unstable/interpolated event names** | `"failed for user_042"` can't be grouped or alerted | fixed names (`"request_failed"`), variables in fields |
| 6 | **Logging PII / secrets** | keys & emails live forever in a searchable store | redact on the **write path**, denylist + value patterns |
| 7 | **No `trace_id` / `session_id`** | logs are a shuffled pile; no drill-down, no replay | attach correlation IDs to every span and log line |
| 8 | **Unbounded log growth** | disk fills; queries slow to a crawl | sample DEBUG, retain/rotate, ship to a real store |
| 9 | **Query on parsed-at-read-time only** | fine at 200 lines, dies at 200M | index in a real backend (Loki/ES/Datadog/Langfuse) |
| 10 | **Naive time filtering (local time / no tz)** | "last hour" is wrong across regions/DST | store UTC ISO-8601; compare tz-aware datetimes |

In [ ]:
# === Verification checklist ==========================================
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    print(("PASS" if cond else "FAIL"), "-", name)

# structured logging
_tb = io.StringIO(); _tl = StructuredLogger(_tb, clock=lambda: NOW)
r0 = _tl.info("evt", user_id="u1", x=1)
check("log record is structured JSON with fields", json.loads(_tb.getvalue())["x"] == 1)
check("message is a stable event name", r0["event"] == "evt")

# context propagation
_cb = io.StringIO(); _cl = StructuredLogger(_cb, clock=lambda: NOW,
                                            context_provider=current_context)
with request_scope(user_id="uX", session_id="sX"):
    _cl.info("in_scope")
check("contextvars auto-attach user_id", json.loads(_cb.getvalue())["user_id"] == "uX")
check("no ambient context after scope exits", _current_ctx.get() is None)

# store + search
store2 = TraceStore(LOG_PATH)
check("store loads spans and logs", len(store2.spans()) > 0 and len(store2.logs()) > 0)
hits2 = store2.logs().user("user_042").failed().last(60)
check("payoff query finds injected incident", len(hits2) > 0)
check("query is specific to user_042", all(r["user_id"] == "user_042" for r in hits2))

# session reconstruction
_sid = Counter(r.get("session_id") for r in store2.logs().rows
               if r.get("session_id")).most_common(1)[0][0]
_tl2 = reconstruct_session(store2, _sid)
_times = [r["ts"] for r in _tl2 if r.get("ts")]
check("session timeline is chronologically ordered", _times == sorted(_times))

# redaction
_rb = io.StringIO(); _rl = RedactingLogger(_rb, clock=lambda: NOW)
_rl.info("x", api_key="sk-live-ZZZ", note="me@x.com")
check("redaction scrubs keys and emails",
      "sk-live" not in _rb.getvalue() and "me@x.com" not in _rb.getvalue())

# module on disk
check("observability/logging_search.py written", os.path.exists(module_path))

passed = sum(1 for _, ok in checks if ok)
print(f"\n{passed}/{len(checks)} checks passed.")
assert passed == len(checks), "some checks failed"
print("All verification checks passed.")

## 10. Summary, homework & what's next

#### What you built today

| Concept | The tool | The takeaway |
|---|---|---|
| **Structured logging** | `StructuredLogger` (JSONL) | log **events with fields**, not sentences — every field is queryable |
| **Correlation IDs** | `trace_id` / `span_id` / `session_id` / `user_id` | the join keys that turn a pile of lines into a graph |
| **Context propagation** | `contextvars` + `request_scope` | attach identity **once**, auto-flows to every span & log — async-safe |
| **Trace search** | `TraceStore` + chainable `Query` | answer *"which requests, for whom"* in one line, not a grep safari |
| **Session reconstruction** | `reconstruct_session()` | replay a full multi-run conversation in time order |
| **PII redaction** | `redact()` on the write path | secrets & emails never reach the searchable store |

You now have the **detect → diagnose** loop complete: L71's metrics tell you *something* broke; L72's search tells you *exactly what*, *for whom*, and lets you *read the story*.

#### Homework (extend the toolkit)
1. **Add a `cost_between(lo, hi)` filter** to `Query` and find the most expensive failing traces — where are you burning money on requests that fail anyway?
2. **Aggregate by field:** write `Query.group_by("user_id")` returning counts, then find the top-3 users by error count *without* the pager telling you who to look at.
3. **Full trace tree:** reconstruct a single `trace_id` into a parent/child **span tree** (use `parent_id`) and pretty-print it as an indented call tree.
4. **Real backend:** pipe the JSONL into **Grafana Loki** (or Langfuse) locally and re-run the payoff query in their UI — feel the difference vs. parse-on-read.
5. **Wire it into `agent-bench`:** make your L61/L70 harness emit one structured log line per task attempt through `RedactingLogger`, then query *"all failed attempts for environment=ShellEnv in this run."*

#### Next lesson — L73: Alerting, SLOs & on-call
You can now *find* an incident after you're paged. L73 closes the loop: **how the page fires in the first place.** We'll define **SLOs** (e.g. "99% of requests succeed", "p95 < 3s"), compute **error budgets**, build **threshold + burn-rate alerts** on top of L71's metrics, and design alerts that page on *real* problems without waking you for noise (hysteresis, for-duration windows, severity routing).

*See you tomorrow. — Your AI tutor* 🎓